## Notebook content

This notebook submits a new AutoML **tabular** pipeline run, monitors progress, and displays the model leaderboard when training completes.

💡 **Tips:**
- Ensure the AutoML pipeline is uploaded to your Kubeflow Pipelines instance before submitting.
- Configure training data and pipeline parameters before submitting.
- Attach an S3 connection to the workbench so artifact download works after the run completes.

### Contents

**[Setup](#setup)**  
**[Experiment configuration](#experiment-configuration)**  
&nbsp;&nbsp;&nbsp;&nbsp;**[Kubeflow connection](#kfp-connection)**  
&nbsp;&nbsp;&nbsp;&nbsp;**[Run defaults](#run-defaults)**  
&nbsp;&nbsp;&nbsp;&nbsp;**[Training data](#training-data)**  
&nbsp;&nbsp;&nbsp;&nbsp;**[Pipeline parameters](#pipeline-parameters)**  
**[Connect to Kubeflow Pipelines](#connect-to-kubeflow-pipelines)**  
**[Discover uploaded pipeline](#discover-uploaded-pipelines)**  
**[Preflight checks](#preflight-checks)**  
**[Submit run](#submit-pipeline-run)**  
**[Monitor run status](#monitor-run-status)**  
**[Review run results](#review-run-results)**  
&nbsp;&nbsp;&nbsp;&nbsp;**[Leaderboard](#leaderboard)**  
**[Summary and next steps](#summary-and-next-steps)**


<a id="setup"></a>
## Setup


### Notebook settings


In [ ]:
import warnings

warnings.filterwarnings("ignore")


### Install dependencies


In [ ]:
%pip install -q "kfp>=2.16" boto3 pandas


### Connection helpers


In [ ]:
import os
from datetime import datetime, timezone
from pathlib import Path

# Kubeflow Pipelines API (OpenShift AI workbench sets KF_PIPELINES_* / ELYRA_RUNTIME_CONFIG)
_K8S_NAMESPACE_PATH = "/var/run/secrets/kubernetes.io/serviceaccount/namespace"


def _read_text_file(path: str) -> str | None:
    try:
        with open(path, encoding="utf-8") as handle:
            value = handle.read().strip()
    except OSError:
        return None
    return value or None


def _pipeline_host_from_elyra_runtime() -> str | None:
    runtime_config = os.environ.get("ELYRA_RUNTIME_CONFIG")
    if not runtime_config:
        return None
    try:
        import json

        with open(runtime_config, encoding="utf-8") as handle:
            payload = json.load(handle)
    except (OSError, json.JSONDecodeError, TypeError):
        return None
    metadata = payload.get("metadata") if isinstance(payload, dict) else None
    if not isinstance(metadata, dict):
        return None
    endpoint = metadata.get("api_endpoint")
    return endpoint.strip() if isinstance(endpoint, str) and endpoint.strip() else None


def _existing_file(path: str | None) -> str | None:
    if not path:
        return None
    candidate = Path(path.strip())
    return str(candidate) if candidate.is_file() else None


def _resolve_kfp_ssl_ca_cert() -> str | None:
    import tempfile

    ca_candidates = [
        os.environ.get("KF_PIPELINES_SSL_SA_CERTS"),
        os.environ.get("PIPELINES_SSL_SA_CERTS"),
        os.environ.get("SSL_CERT_FILE"),
        os.environ.get("REQUESTS_CA_BUNDLE"),
        os.environ.get("GIT_SSL_CAINFO"),
        "/var/run/secrets/kubernetes.io/serviceaccount/ca.crt",
    ]
    existing_paths: list[str] = []
    seen: set[str] = set()
    for raw_path in ca_candidates:
        path = _existing_file(raw_path)
        if path and path not in seen:
            seen.add(path)
            existing_paths.append(path)
    if not existing_paths:
        return None
    if len(existing_paths) == 1:
        return existing_paths[0]
    merged = tempfile.NamedTemporaryFile(
        mode="w",
        suffix="-kfp-ca-bundle.crt",
        delete=False,
        encoding="utf-8",
    )
    for path in existing_paths:
        merged.write(Path(path).read_text(encoding="utf-8").rstrip("\n"))
        merged.write("\n")
    merged.close()
    return merged.name


<a id="experiment-configuration"></a>
## Experiment configuration

Review platform, data, and pipeline settings before submitting a new run. Platform values are resolved from the workbench environment; training data and AutoML parameters are pre-filled from the pipeline run that generated this notebook.


<a id="kfp-connection"></a>
### Kubeflow connection

KFP API endpoint, namespace, token, and TLS settings. Usually resolved from the workbench environment.

If the endpoint is not available, get the host for the `ds-pipeline...` route in your RHOAI project:

```bash
oc get route -n <RHOAI_PROJECT_NAME>
```

Use the route host as `kfp_host`, for example: `https://ds-pipeline-<project>.apps.some-cluster.aws.rh-ods.com/`.


In [ ]:
import os

if "_pipeline_host_from_elyra_runtime" not in globals():
    raise RuntimeError("Run Setup > Connection helpers before configuring Kubeflow connection.")

kfp_host = (
    os.environ.get("KF_PIPELINES_ENDPOINT")
    or os.environ.get("PIPELINE_HOST")
    or _pipeline_host_from_elyra_runtime()
    or os.environ.get("KFP_HOST")
    or "https://<REPLACE_KFP_HOST>"
)
kfp_namespace = (
    os.environ.get("KFP_NAMESPACE")
    or _read_text_file(_K8S_NAMESPACE_PATH)
    or "<REPLACE_NAMESPACE>"
)
kfp_token = os.environ.get("KFP_TOKEN")
kfp_sa_token_path = os.environ.get("KF_PIPELINES_SA_TOKEN_PATH")
kfp_ssl_ca_cert = _resolve_kfp_ssl_ca_cert()
kfp_verify_ssl = os.environ.get("KFP_VERIFY_SSL", "true").lower() not in {"0", "false", "no"}

_missing = []
if not kfp_host or "<REPLACE_" in kfp_host:
    _missing.append("kfp_host")
if not kfp_namespace or "<REPLACE_" in kfp_namespace:
    _missing.append("kfp_namespace")
if not kfp_token and not kfp_sa_token_path:
    _missing.append("kfp_token (or KF_PIPELINES_SA_TOKEN_PATH)")
if _missing:
    print("Set in this cell, then re-run:", ", ".join(_missing))


<a id="run-defaults"></a>
### Run defaults

Pipeline name, artifact storage, experiment, optional version pin, submission confirmation, and monitoring timeouts.

To use a specific RHOAI release, copy its `version_id` from **Discover uploaded pipeline** into `pipeline_version_id`; leave it empty to use the latest version. OpenShift AI dashboard links are detected automatically; set `RHOAI_DASHBOARD_URL` only to override that URL.


In [ ]:
pipeline_name = "autogluon-tabular-training-pipeline"

# Artifact store bucket (pipeline outputs are written under <pipeline_name>/<run_id>/)
artifacts_bucket = os.environ.get("AWS_S3_BUCKET", "<REPLACE_ARTIFACTS_BUCKET>")

experiment_name = os.environ.get(
    "KF_PIPELINES_DEFAULT_EXPERIMENT_NAME",
    os.environ.get("KFP_EXPERIMENT_NAME", "automl-experiments"),
)

pipeline_version_id = ""

kfp_ui_base_url = os.environ.get("KFP_UI_BASE_URL", kfp_host).rstrip("/")
rhoai_dashboard_url = os.environ.get("RHOAI_DASHBOARD_URL", "").rstrip("/")

run_timeout_seconds = int(os.environ.get("KFP_RUN_TIMEOUT", "3600"))
poll_interval_seconds = 30
status_heartbeat_interval_seconds = 300

<a id="training-data"></a>
### Training data

S3 locations for training data and optional external test data.


In [ ]:
train_data_secret_name = <REPLACE_S3_SECRET>
train_data_bucket_name = <REPLACE_DATA_BUCKET>
train_data_file_key = <REPLACE_DATA_FILE_KEY>

# Optional user-provided test dataset (leave both empty for internal holdout split)
test_data_bucket_name = <REPLACE_TEST_DATA_BUCKET>
test_data_file_key = <REPLACE_TEST_DATA_FILE_KEY>

<a id="pipeline-parameters"></a>
### Pipeline parameters

AutoML tabular settings passed to `autogluon-tabular-training-pipeline`.


In [ ]:
label_column = <REPLACE_LABEL_COLUMN>
task_type = <REPLACE_TASK_TYPE>  # binary | multiclass | regression
top_n = <REPLACE_TOP_N>
positive_class = <REPLACE_POSITIVE_CLASS>
eval_metric = <REPLACE_EVAL_METRIC>
preset = <REPLACE_PRESET>  # speed | balanced

<a id="connect-to-kubeflow-pipelines"></a>
## Connect to Kubeflow Pipelines

Create a `kfp.Client` using the endpoint, namespace, credentials, and TLS settings configured above. The client is used by the following cells to discover pipeline versions, submit the run, and monitor its status.


In [ ]:
import kfp
from urllib.parse import urlparse

if kfp_token and (urlparse(kfp_host).scheme != "https" or not kfp_verify_ssl):
    raise ValueError(
        "KFP_TOKEN requires an HTTPS KFP host with certificate verification enabled."
    )

client_kwargs = {
    "host": kfp_host if kfp_host.endswith("/") else f"{kfp_host}/",
    "namespace": kfp_namespace,
    "verify_ssl": kfp_verify_ssl,
}
if kfp_ssl_ca_cert:
    client_kwargs["ssl_ca_cert"] = kfp_ssl_ca_cert

bearer_token = kfp_token
if not bearer_token and kfp_sa_token_path:
    bearer_token = _read_text_file(kfp_sa_token_path)
if bearer_token:
    client_kwargs["existing_token"] = bearer_token

client = kfp.Client(**client_kwargs)
print(f"Connected to KFP in namespace: {client.get_user_namespace()}")


<a id="discover-uploaded-pipelines"></a>
## Discover uploaded pipeline

List pipeline versions uploaded to Kubeflow Pipelines. By default this is filtered to `pipeline_name` from Experiment configuration.


In [ ]:
import json

import pandas as pd
from IPython.display import display

_SORT = "created_at desc"


def _pipeline_filter(name):
    return json.dumps({"predicates": [{"operation": 1, "key": "display_name", "stringValue": name}]})


def get_pipeline_and_versions(name):
    pipelines = client.list_pipelines(
        page_size=50,
        sort_by=_SORT,
        filter=_pipeline_filter(name),
    ).pipelines or []
    if not pipelines:
        raise RuntimeError(f"Pipeline '{name}' not found. Upload it to this cluster first.")
    if len(pipelines) > 1:
        raise RuntimeError(f"Multiple pipelines named '{name}'.")
    pipeline = pipelines[0]
    versions = client.list_pipeline_versions(
        pipeline_id=pipeline.pipeline_id, page_size=50, sort_by=_SORT
    ).pipeline_versions or []
    return pipeline, versions


def resolve_pipeline_template(name, version_id=None):
    pipeline, versions = get_pipeline_and_versions(name)
    if version_id:
        return pipeline.pipeline_id, version_id
    if not versions:
        raise RuntimeError(f"Pipeline '{name}' has no uploaded versions.")
    return pipeline.pipeline_id, versions[0].pipeline_version_id


pipeline, pipeline_versions = get_pipeline_and_versions(pipeline_name)
pipeline_rows = [
    {
        "pipeline": pipeline.display_name,
        "pipeline_id": pipeline.pipeline_id,
        "version_id": version.pipeline_version_id,
        "version_name": version.display_name,
        "created_at": version.created_at,
    }
    for version in pipeline_versions
]
display(pd.DataFrame(pipeline_rows)) if pipeline_rows else print(f"No versions for {pipeline_name!r}.")


<a id="preflight-checks"></a>
## Preflight checks

Validate the selected pipeline, required configuration, and access to the artifact bucket before starting a potentially long training run.


In [ ]:
import boto3
from botocore.exceptions import BotoCoreError, ClientError


def _is_missing(value):
    return not str(value).strip() or "<REPLACE_" in str(value)


required_values = {
    "artifacts_bucket": artifacts_bucket,
    "train_data_secret_name": train_data_secret_name,
    "train_data_bucket_name": train_data_bucket_name,
    "train_data_file_key": train_data_file_key,
    "label_column": label_column,
    "task_type": task_type,
    "top_n": top_n,
    "eval_metric": eval_metric,
    "preset": preset,
}
missing_values = [name for name, value in required_values.items() if _is_missing(value)]
if missing_values:
    raise ValueError(f"Set required configuration values: {', '.join(missing_values)}")
pipeline, pipeline_versions = get_pipeline_and_versions(pipeline_name)
if not pipeline_versions:
    raise RuntimeError(f"Pipeline '{pipeline_name}' has no uploaded versions.")
if pipeline_version_id and not any(
    version.pipeline_version_id == pipeline_version_id for version in pipeline_versions
):
    raise ValueError(f"Version '{pipeline_version_id}' does not belong to '{pipeline_name}'.")

missing_s3_environment = [
    name for name in ("AWS_S3_ENDPOINT", "AWS_ACCESS_KEY_ID") if not os.environ.get(name, "").strip()
]
if missing_s3_environment:
    raise ValueError(f"Attach an S3 connection; missing: {', '.join(missing_s3_environment)}")

s3_preflight = boto3.client(
    "s3",
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ.get("AWS_SECRET_ACCESS_KEY", ""),
    region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
    verify=os.environ.get("SSL_CERT_FILE") or os.environ.get("REQUESTS_CA_BUNDLE") or True,
)
try:
    s3_preflight.head_bucket(Bucket=artifacts_bucket)
except (BotoCoreError, ClientError) as exc:
    raise RuntimeError(f"Cannot access artifact bucket '{artifacts_bucket}'.") from exc

selected_version_id = pipeline_version_id or pipeline_versions[0].pipeline_version_id
print(f"Preflight passed: {pipeline_name} / {selected_version_id} / {artifacts_bucket}")


<a id="submit-pipeline-run"></a>
## Submit run

Submit a new run using the pipeline already uploaded to Kubeflow Pipelines.


In [ ]:
pipeline_arguments = {
    "train_data_secret_name": train_data_secret_name,
    "train_data_bucket_name": train_data_bucket_name,
    "train_data_file_key": train_data_file_key,
    "test_data_bucket_name": test_data_bucket_name,
    "test_data_file_key": test_data_file_key,
    "label_column": label_column,
    "task_type": task_type,
    "top_n": top_n,
    "positive_class": positive_class,
    "eval_metric": eval_metric,
    "preset": preset,
}

pipeline_id, resolved_version_id = resolve_pipeline_template(
    pipeline_name,
    pipeline_version_id or None,
)

experiment = client.create_experiment(name=experiment_name)
run_name = f"automl-tabular-experiment-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}"

run_kwargs = {
    "experiment_id": experiment.experiment_id,
    "job_name": run_name,
    "pipeline_id": pipeline_id,
    "version_id": resolved_version_id,
    "params": pipeline_arguments,
}

run_info = client.run_pipeline(**run_kwargs)
run_id = run_info.run_id
print(f"Submitted run: {run_name}")
print(f"Run ID: {run_id}")
print(f"Pipeline: {pipeline_name}")
print(f"Pipeline version: {resolved_version_id}")

from IPython.display import Markdown, display
from urllib.parse import quote, urlparse

kfp_host_info = urlparse(kfp_host)
kfp_hostname = kfp_host_info.hostname or ""
dspa_prefix = f"ds-pipeline-dspa-{kfp_namespace}."
run_path = f"/develop-train/pipelines/runs/{quote(kfp_namespace)}/runs/{quote(run_id)}"
if rhoai_dashboard_url:
    run_url = f"{rhoai_dashboard_url}{run_path}"
elif kfp_hostname.startswith(dspa_prefix):
    dashboard_host = f"rh-ai.{kfp_hostname.removeprefix(dspa_prefix)}"
    run_url = f"{kfp_host_info.scheme or 'https'}://{dashboard_host}{run_path}"
else:
    run_url = f"{kfp_ui_base_url}/#/runs/details/{run_id}"
display(Markdown(f"[Open this run in Kubeflow Pipelines]({run_url})"))

<a id="monitor-run-status"></a>
## Monitor run status

Poll the run until it reaches a terminal state. Status changes are printed immediately; an unchanged run receives a periodic heartbeat.


In [ ]:
import time


def _run_state(run_detail) -> str:
    run = getattr(run_detail, "run", run_detail)
    state = getattr(run, "state", None)
    if state is None and hasattr(run, "status"):
        state = getattr(run.status, "state", None)
    return str(state or "UNKNOWN").upper()


def _format_elapsed(seconds: float) -> str:
    total_seconds = int(seconds)
    minutes, seconds = divmod(total_seconds, 60)
    hours, minutes = divmod(minutes, 60)
    if hours:
        return f"{hours}h {minutes}m {seconds}s"
    if minutes:
        return f"{minutes}m {seconds}s"
    return f"{seconds}s"


def poll_run_status(
    client, run_id: str, timeout_seconds: int, interval_seconds: int = 30, heartbeat_interval_seconds: int = 300
):
    terminal = {"SUCCEEDED", "FAILED", "CANCELED", "SKIPPED"}
    deadline = time.time() + timeout_seconds
    started_at = time.monotonic()
    last_state = None
    last_heartbeat_at = started_at
    while time.time() < deadline:
        detail = client.get_run(run_id)
        state = _run_state(detail)
        now = time.monotonic()
        elapsed = _format_elapsed(now - started_at)
        checked_at = datetime.now(timezone.utc).strftime("%H:%M:%S UTC")
        if state != last_state:
            print(f"[{checked_at}] {state} — elapsed {elapsed}")
            last_state = state
            last_heartbeat_at = now
        elif now - last_heartbeat_at >= heartbeat_interval_seconds:
            print(f"[{checked_at}] still {state} — elapsed {elapsed}")
            last_heartbeat_at = now
        if state in terminal:
            return detail, state
        time.sleep(interval_seconds)
    raise TimeoutError(f"Run {run_id} did not finish within {timeout_seconds}s")


_, final_state = poll_run_status(
    client,
    run_id,
    timeout_seconds=run_timeout_seconds,
    interval_seconds=poll_interval_seconds,
    heartbeat_interval_seconds=status_heartbeat_interval_seconds,
)
print(f"Final state: {final_state}")
if final_state != "SUCCEEDED":
    raise RuntimeError(f"Pipeline run {run_id} finished with state {final_state}")


<a id="review-run-results"></a>
## Review run results

Download run-level artifacts from `s3://<artifacts_bucket>/<pipeline_name>/<run_id>/`.


In [ ]:
from pathlib import Path, PurePosixPath

import boto3

required_env_vars = (
    "AWS_S3_ENDPOINT",
    "AWS_ACCESS_KEY_ID",
)
missing_vars = [name for name in required_env_vars if not os.environ.get(name, "").strip()]
if missing_vars:
    raise ValueError(
        f"Missing required environment variable(s): {', '.join(missing_vars)}. "
        "Attach your S3 connection to this workbench and try again."
    )

s3_verify = (
    os.environ.get("SSL_CERT_FILE")
    or os.environ.get("REQUESTS_CA_BUNDLE")
    or True
)


def list_run_object_keys(s3, bucket: str, prefix: str) -> list[str]:
    keys: list[str] = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents") or []:
            key = obj["Key"]
            if not key.endswith("/"):
                keys.append(key)
    return sorted(keys)


def download_keys(s3, bucket: str, prefix: str, keys: list[str], dest_dir: Path) -> list[Path]:
    downloaded: list[Path] = []
    dest_root = dest_dir.resolve()
    prefix_path = PurePosixPath(prefix)
    for key in keys:
        relative = PurePosixPath(key).relative_to(prefix_path)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError(f"Unsafe artifact key: {key!r}")
        target = (dest_dir / relative).resolve()
        if not target.is_relative_to(dest_root):
            raise ValueError(f"Refusing to write outside {dest_dir}: {key}")
        target.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(bucket, key, str(target))
        downloaded.append(target)
    return downloaded


artifact_prefix = f"{pipeline_name}/{run_id}/"
local_output_dir = Path("automl_run_artifacts") / run_id
local_output_dir.mkdir(parents=True, exist_ok=True)

s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ.get("AWS_SECRET_ACCESS_KEY", ""),
    region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
    verify=s3_verify,
)
artifact_keys = list_run_object_keys(s3, artifacts_bucket, artifact_prefix)

interesting_suffixes = (
    "component_status.json",
    "html_artifact",
    "model.json",
)
selected_keys = [
    key
    for key in artifact_keys
    if any(key.endswith(suffix) for suffix in interesting_suffixes)
]

downloaded_paths = download_keys(s3, artifacts_bucket, artifact_prefix, selected_keys, local_output_dir)

print(f"Downloaded {len(downloaded_paths)} file(s) to {local_output_dir.resolve()}")

<a id="leaderboard"></a>
### Leaderboard

Review ranked refitted models from this run.


In [ ]:
from IPython.display import HTML, display

leaderboard_paths = sorted(local_output_dir.rglob("html_artifact"))
if not leaderboard_paths:
    raise FileNotFoundError(
        f"Leaderboard artifact not found under {local_output_dir}. "
        "Ensure the run succeeded and the S3 connection can access pipeline outputs."
    )

leaderboard_html = leaderboard_paths[0].read_text(encoding="utf-8")
leaderboard_html = leaderboard_html.replace(
    "</head>",
    """<style>
        .main-content { max-width: 100% !important; }
        .table-scroll { overflow-x: auto !important; }
        .table-scroll table { width: max-content !important; min-width: 100% !important; }
        .table-scroll th, .table-scroll td { white-space: nowrap; }
    </style></head>""",
)
display(HTML(leaderboard_html))


<a id="summary-and-next-steps"></a>
## Summary and next steps

You submitted a new AutoML tabular pipeline run and reviewed its leaderboard.

**Next steps:**
1. Pick a model from the leaderboard table.
2. Open its predictor notebook from `models_artifact/<ModelName>_FULL/notebooks/automl_predictor_notebook.ipynb` for local inference and model analysis.
3. Deploy the chosen model online with the [KServe AutoGluon runtime](https://github.com/kserve/kserve/tree/master/python/autogluonserver) (`modelFormat: autogluon`):
   - Storage URI: `models_artifact/<ModelName>_FULL/predictor/`
   - Request schema: use the `inference` block in `models_artifact/<ModelName>_FULL/model.json` to build the `POST /v1/models/{name}:predict` body.
   - In OpenShift AI, create an InferenceService that references the predictor storage URI and the AutoGluon ServingRuntime.